# Max Circuits Per Second (MCPS)

MCPS is the device-level speed ceiling defined in Sec. II.B of A. Wack, *CLOPS: Benchmarking System Speed at Utility
Scale*, arXiv:2608.18044:

$$\mathrm{MCPS} = \frac{1}{t_\mathrm{gate} + t_\mathrm{sp} + t_\mathrm{m} + t_\mathrm{delay}}$$

where $t_\mathrm{gate}$ is the duration of a Hadamard gate on all qubits in parallel, $t_\mathrm{sp}$ the state
preparation time, $t_\mathrm{m}$ the measurement time and $t_\mathrm{delay}$ the delay between circuits, i.e. the reset.
No circuit containing two-qubit gates can run faster, so MCPS bounds CLOPS from above.

The benchmark repeatedly runs a single layer of Hadamard gates on all qubits followed by a measurement, and reports:

* `mcps_measured`: shots per second between the start and the end of the execution on the device, averaged over the repeated jobs.
* `mcps_ceiling`: one over the duration of a shot, taken from the compiled schedule (gate, measurement and reset) plus the end delay.
* `mcps_wall_clock`: shots per second over the whole job, from its reception to its results being ready.

## Connect to the backend

In [ ]:
import os

from iqm.qiskit_iqm import IQMProvider

os.environ["IQM_TOKEN"] = ""  # provide your token
quantum_computer = ""  # provide the quantum computer name, e.g. "emerald", "garnet", "sirius"
iqm_server_url = "https://resonance.iqm.tech/"  # provide the IQM server URL

provider = IQMProvider(iqm_server_url, quantum_computer=quantum_computer)
backend = provider.get_backend()

## Circuit-level MCPS

By default all qubits of the backend are used, since MCPS is a device-level metric. Set `qubits` to benchmark a subset.
Many shots per job are used so that the overhead of each job does not affect the measured rate.

In [ ]:
from iqm.benchmarks.quantum_volume.mcps import MCPSBenchmark, MCPSConfiguration

EXAMPLE_MCPS = MCPSConfiguration(shots=10_000, num_repetitions=5)

benchmark_mcps = MCPSBenchmark(backend, EXAMPLE_MCPS)
run_mcps = benchmark_mcps.run()
result_mcps = benchmark_mcps.analyze()
result_mcps.observations

In [ ]:
result_mcps.plot_all()

## Pulse-level MCPS

At pulse level the circuit is compiled on the client and the compiler settings can be changed through `pulse_settings`,
a dictionary from a dotted path in the settings tree to the value to set. The reset, $t_\mathrm{delay}$ in the formula
above, can be done in two ways.

### Passive reset

By default the qubits are reset by waiting for them to relax. The wait duration of every qubit and coupler is set as
follows (paths of qubits that are not benchmarked are skipped):

In [ ]:
from iqm.benchmarks.quantum_volume.mcps import MCPSPulseBenchmark, MCPSPulseConfiguration

settings = backend.client.get_standard_compiler().get_settings([])

def reset_wait(duration: float) -> dict[str, float]:
    return {
        f"gates.reset_wait.reset_wait.{locus}.duration": duration
        for locus, _ in settings.gates.reset_wait.reset_wait.child_nodes
    }

results = {}
for duration in (400e-6, 200e-6, 100e-6):
    config = MCPSPulseConfiguration(shots=10_000, num_repetitions=5, pulse_settings=reset_wait(duration))
    benchmark = MCPSPulseBenchmark(backend, config)
    benchmark.run()
    results[f"passive {duration * 1e6:.0f} us"] = benchmark.analyze()

### Active reset

With `active_reset_cycles` the qubits are measured and flipped back to the ground state instead of waiting, and the
reset wait is not used.

In [ ]:
config = MCPSPulseConfiguration(shots=10_000, num_repetitions=5, active_reset_cycles=1)
benchmark = MCPSPulseBenchmark(backend, config)
benchmark.run()
results["active 1 cycle"] = benchmark.analyze()

## Comparison

A shorter reset raises the ceiling. The measured MCPS shows how close the device gets to it.

In [ ]:
for label, result in results.items():
    values = {o.name: o.value for o in result.observations}
    print(
        f"{label:>16}: measured {values.get('mcps_measured', float('nan')):8.0f}"
        f"  ceiling {values.get('mcps_ceiling', float('nan')):8.0f}"
        f"  wall clock {values.get('mcps_wall_clock', float('nan')):8.0f}"
    )